# 교차검증 실습

**Cross-validation · CV**

데이터를 여러 번 나누어 학습과 평가를 반복하는 검증 방법.

소재 분야에서 이해하기: 관련 시료는 같은 그룹으로 묶어 평가 정보가 새지 않게 한다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [scikit-learn 교차검증 문서](https://scikit-learn.org/stable/modules/cross_validation.html)

## 1. 한 번의 분할은 운에 좌우됩니다

분할 방법을 바꿔가며 점수가 얼마나 흔들리는지 봅니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

def make_alloy_data(n=240, noise=6.0, seed=0):
    """개념 확인용 합성 데이터. 실제 합금 측정값이 아닙니다.

    x1 소성 온도(600-900 C), x2 유지 시간(0.5-8 h), x3 첨가 원소 비율(0-5 at%),
    x4 측정 노이즈만 담긴 무의미한 변수. y 는 경도(HV) 를 흉내낸 값입니다.
    """
    rng = np.random.default_rng(seed)
    x1 = rng.uniform(600, 900, n)
    x2 = rng.uniform(0.5, 8.0, n)
    x3 = rng.uniform(0.0, 5.0, n)
    x4 = rng.normal(0.0, 1.0, n)
    y = (120 + 0.14 * (x1 - 600) + 9.0 * np.sqrt(x2) + 11.0 * x3
         - 0.9 * x3 ** 2 - 0.004 * (x1 - 750) * x2 + rng.normal(0, noise, n))
    X = np.column_stack([x1, x2, x3, x4])
    return X, y, ['소성온도', '유지시간', '첨가비율', '무관변수']


X, y, FEATURES = make_alloy_data()
print(X.shape, y.shape, FEATURES)
print('경도 평균 %.1f, 표준편차 %.1f' % (y.mean(), y.std()))

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.ensemble import RandomForestRegressor

single = [train_test_split(X, y, test_size=0.25, random_state=seed) for seed in range(6)]
for seed, (X_tr, X_te, y_tr, y_te) in enumerate(single):
    model = RandomForestRegressor(n_estimators=200, random_state=0).fit(X_tr, y_tr)
    print('분할 seed %d -> R2 %.3f' % (seed, model.score(X_te, y_te)))

scores = cross_val_score(RandomForestRegressor(n_estimators=200, random_state=0), X, y,
                         cv=KFold(5, shuffle=True, random_state=0), scoring='r2')
print('\n5겹 교차검증 R2 %.3f ± %.3f' % (scores.mean(), scores.std()))

## 2. 관련 시료가 흩어지면 점수가 부풀려집니다

같은 배치에서 나온 시료를 학습과 검증에 나눠 담으면 정보가 새어 성능이 좋아 보입니다.

In [ ]:
from sklearn.model_selection import GroupKFold

batch = np.repeat(np.arange(len(X) // 8), 8)[:len(X)]
batch_effect = rng.normal(0, 8, batch.max() + 1)[batch]
y_batched = y + batch_effect

plain = cross_val_score(RandomForestRegressor(n_estimators=200, random_state=0), X, y_batched,
                        cv=KFold(5, shuffle=True, random_state=0), scoring='r2').mean()
grouped = cross_val_score(RandomForestRegressor(n_estimators=200, random_state=0), X, y_batched,
                          cv=GroupKFold(5), groups=batch, scoring='r2').mean()
print('배치를 섞은 교차검증 R2 %.3f (부풀려짐)' % plain)
print('배치를 묶은 GroupKFold R2 %.3f (정직한 추정)' % grouped)

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#cross-validation)을 여세요.